# 전체 데이터 Oracle 적재
Managed Folder ID와 Oracle 연결 이름을 설정하고 inspect → restore → check → publish → status 순서로 실행합니다.
전체 원문·자산·조회 데이터를 적재하며 테이블 재생성과 활성 포인터 전환은 하지 않습니다.

In [ ]:
# Dataiku의 새 Python 노트북에 전체를 붙여 실행합니다. SQL 노트북/b​​ash용이 아닙니다.
# 개발계: 코드 ZIP + 기존 genos-rag-data ZIP 두 개를 Managed Folder에 업로드합니다.
# 각 파일을 폴더 최상위에 올리고 아래 ID/연결명을 설정하세요.
ACTION = "inspect"                   # inspect → restore → check → publish → status
INPUT_MODE = "zip"                  # 개발계 zip / 운영계 parts
DATA_ZIP_NAME = "genos-rag-data-20261007T002029Z-7c0b050f_verified.zip"  # 실제 업로드 파일명
EXPECTED_ZIP_SHA256 = ""             # 기존 .sha256 값이 있으면 지정; 없으면 파일별 검증+새 ZIP 지문 기록
FOLDER_ID = "여기에_Managed_Folder_ID"
ORACLE_CONNECTION = "여기에_Oracle_연결_이름"  # SQL 노트북에서 선택한 연결의 정확한 이름
WORK_DIR = "./legal-existing-zip-work"     # 재실행/커널 재시작에도 같은 영속 폴더 사용
PUBLISH_SECONDS = 3600                # 1회 1시간, 최초 7일/누적100만 SQL 한도 유지

import hashlib
import importlib.util
import json
import os
from pathlib import Path
import shutil
import sys
import zipfile
import time
import dataiku

_CODE_ZIP = "LEGAL_ORACLE_EXISTING_ZIP_20261020_R1.zip"
_CODE_PREFIX = "LEGAL_ORACLE_EXISTING_ZIP_20261020_R1/"
_PINS_HASH = "3964c7346d2a9b15ccf6f8c063cd2d6de6f37a28fe7765e8306f326ca40d2431"

def _download(folder,name,destination,*,expected_sha=None,max_bytes=None):
    if destination.is_symlink():raise RuntimeError('입력 경로가 링크입니다: '+str(destination))
    if destination.exists():
        h=hashlib.sha256()
        with destination.open('rb') as f:
            while b:=f.read(1024**2):h.update(b)
        if expected_sha is not None and h.hexdigest()==expected_sha:return
        if expected_sha is None:return  # Code ZIP is independently checked against pinned executable members.
        raise RuntimeError('기존 파일 해시가 다릅니다. 기록을 보존하고 원본 파일을 확인하세요: '+name)
    temporary=destination.with_name(destination.name+'.partial')
    if temporary.exists():
        # Preserve prior interrupted bytes; never use them as completed input.
        temporary=destination.with_name(destination.name+'.partial-'+os.urandom(6).hex())
    h=hashlib.sha256();count=0;last=time.monotonic()
    with folder.get_download_stream('/'+name) as source,temporary.open('xb') as target:
        while b:=source.read(1024**2):
            count+=len(b)
            if max_bytes is not None and count>max_bytes:raise RuntimeError('파일 크기 제한 초과: '+name)
            if shutil.disk_usage(destination.parent).free<512*1024**2:raise RuntimeError('가용 디스크 512MiB 미만. 부분 파일은 보존합니다.')
            h.update(b);target.write(b)
            if time.monotonic()-last>=15:
                print("DOWNLOAD",name,"bytes=",count,"free_bytes=",shutil.disk_usage(destination.parent).free,flush=True);last=time.monotonic()
        target.flush();os.fsync(target.fileno())
    if expected_sha is not None and h.hexdigest()!=expected_sha:raise RuntimeError('다운로드 해시 불일치: '+name)
    temporary.rename(destination)


def _prepare(folder,work):
    work.mkdir(parents=True,exist_ok=True)
    _download(folder,_CODE_ZIP,work/_CODE_ZIP,max_bytes=4*1024**2)
    code=work/'code';code.mkdir(exist_ok=True)
    with zipfile.ZipFile(work/_CODE_ZIP) as z:
        member=_CODE_PREFIX+'CODE_PINS.json'
        if z.getinfo(member).file_size>1024**2:raise RuntimeError('코드 목록 크기 오류')
        raw=z.read(member)
        if hashlib.sha256(raw).hexdigest()!=_PINS_HASH:raise RuntimeError('코드 ZIP 버전/해시 불일치')
        pins=json.loads(raw)
        for relative,sha in pins.items():
            rel=Path(relative)
            if rel.is_absolute() or '..' in rel.parts or rel.suffix!='.py':raise RuntimeError('코드 경로 오류')
            info=z.getinfo(_CODE_PREFIX+relative)
            if info.file_size>4*1024**2:raise RuntimeError('코드 파일 크기 오류')
            data=z.read(info)
            if hashlib.sha256(data).hexdigest()!=sha:raise RuntimeError('코드 파일 해시 오류: '+relative)
            target=code/relative;target.parent.mkdir(parents=True,exist_ok=True)
            if target.is_symlink() or not target.resolve().is_relative_to(code.resolve()):raise RuntimeError('코드 경로 링크 오류')
            if target.exists():
                if target.read_bytes()!=data:raise RuntimeError('기존 실행 코드가 다릅니다: '+relative)
            else:target.write_bytes(data)
        (code/'CODE_PINS.json').write_bytes(raw)
    spec=importlib.util.spec_from_file_location('query_transport_entry',code/'entry.py')
    entry=importlib.util.module_from_spec(spec);spec.loader.exec_module(entry)
    return entry.load()


def _run():
    if ACTION not in {'inspect','restore','check','publish','status'}:raise RuntimeError('ACTION: inspect / restore / check / publish / status')
    if not FOLDER_ID or FOLDER_ID.startswith('여기에_'):raise RuntimeError('FOLDER_ID를 설정하세요.')
    folder=dataiku.Folder(FOLDER_ID);work=Path(WORK_DIR).resolve()
    pack=_prepare(folder,work)
    if ACTION in {'inspect','restore'}:
        if INPUT_MODE=='zip':
            if Path(DATA_ZIP_NAME).name!=DATA_ZIP_NAME or not DATA_ZIP_NAME.endswith('.zip'):raise RuntimeError('DATA_ZIP_NAME은 업로드한 ZIP 파일명만 입력하세요.')
            archive=work/DATA_ZIP_NAME
            _download(folder,DATA_ZIP_NAME,archive,expected_sha=EXPECTED_ZIP_SHA256 or None)
        elif INPUT_MODE=='parts':
            parts=work/'parts';parts.mkdir(exist_ok=True)
            _download(folder,'parts-manifest.json',parts/'parts-manifest.json',max_bytes=32*1024**2)
            info=pack.transport.bounded_json(parts/'parts-manifest.json');pack.validate_parts(info)
            for i,item in enumerate(info['parts'],1):
                _download(folder,item['name'],parts/item['name'],expected_sha=item['sha256'],max_bytes=item['bytes'])
                print('조각 확인:',i,'/',len(info['parts']),flush=True)
            archive=Path(pack.join(parts,work/'joined'))
        else:raise RuntimeError('INPUT_MODE: zip / parts')
        if ACTION=='inspect':
            result=pack.inspect_archive(archive);result.pop('metadata');result.pop('members')
            print('ZIP에 기록된 세대를 확인했습니다. 최신 데이터인지/Oracle 적재 완료인지는 별도입니다.')
            return result
        return pack.restore(archive,work/'restored',expected_sha=EXPECTED_ZIP_SHA256 or None)
    if not ORACLE_CONNECTION or ORACLE_CONNECTION.startswith('여기에_'):raise RuntimeError('ORACLE_CONNECTION을 설정하세요.')
    from lawrag.pipeline.oracle_query_dataiku import DataikuClient
    client=DataikuClient(dataiku.SQLExecutor2(connection=ORACLE_CONNECTION),ORACLE_CONNECTION,dataiku.default_project_key())
    binding,target=client.identity();print('Oracle 연결 대상:',json.dumps(target,ensure_ascii=False))
    restored=work/'restored'
    if not (restored/'full-manifest.json').is_file():raise RuntimeError('먼저 ACTION="restore"를 완료하세요.')
    logs=work/'oracle-import'
    print('실행 기록:',str(logs),flush=True)
    return pack.import_generation(client,restored,target_binding=binding,directory=logs,action=ACTION,seconds=PUBLISH_SECONDS)


FULL_TRANSFER_RESULT = None
try:
    FULL_TRANSFER_RESULT = _run()
    print(json.dumps(FULL_TRANSFER_RESULT,ensure_ascii=False,indent=2))
    print('전체 원문·자산·조회 데이터 이관입니다. ACTIVE 전환·STATE 이력 이관·서빙 배포는 별도입니다.')
except Exception as exc:
    # Dataiku platform exceptions may contain connection details. Package exceptions are already sanitized.
    known=isinstance(exc,RuntimeError)
    message=str(exc) if known else type(exc).__name__
    print('중단:',message)
    raise RuntimeError(message) from None
